Here, I want to explore the potentials of: "Contextual LCIA without the overhead: an exchange-based framework for flexible impact assessment"'

- To install Edges, install from Anaconda Prompt in your activa environment via pip install edges

In [1]:
# basic packages from brightway
import bw2analyzer as ba
import bw2calc as bc
import bw2data as bd
import bw2io as bi
import pandas as pd

# import edges packages
from edges import EdgeLCIA
import edges

11:02:53+0200 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


In [2]:
# call the project we want to work in, in this case brightway25 where we have set up the databases
bd.projects.set_current('LCA_Toolbox')

In [3]:
bd.databases

Databases dictionary with 8 object(s):
	BONSAI_V2.1.6
	BONSAI_V2.1.6 biosphere
	bafu
	biosphere3
	ecoinvent-3.10-biosphere
	ecoinvent-3.12-biosphere
	ecoinvent-3.12-consequential
	template_consequential

In [4]:
ecoinvent = bd.Database('ecoinvent-3.12-consequential')
bafu = bd.Database('bafu')

In [5]:
len(bafu)

11747

Notes from Knowledge Sharing
- Built on Romain Sacchi's publications and research
- To learn more about assessing raw material criticality, I need to also look into GeoPolRisk
- BW2 regional includes potential to include GIS
- Edges > BW2 regional: Edges has more methods, critical raw material assessments, can adjust CFs to technosphere exchanges, where regionalised BW2 can only do biosphere CF exchange modification
- GeoPolRisk is a method for criticality assessments that provides a factor for criticality

Questions
- GeoPolRisk package as part of Edges or what are the required package constructs to work with assessing raw material criticality?
- How are CFs adjusted in GeoPolRisk?
- So do the main benefits of Edges trace back to more disaggregated background data for regionalised characterisation factors? So how were these 'improved' CFs defined and extracted?
- Where and how to install the functions in the first place?

In [6]:
milk_qc = ecoinvent.get(name='milk production, from cow', location='CA-QC')

In [7]:
iw_water = ('ecoinvent-3.12',
  'IMPACT World+ v2.1, footprint version',
  'water use',
  'water scarcity footprint')

Conventional LCA

In [8]:
# Quick LCIA calculation
milk_qc_lca = milk_qc.lca(iw_water)
milk_qc_lca.score

1.2726834698937135

In [9]:
milk_qc_lca.to_dataframe().pivot_table(index=['col_name','col_location'],values='amount',aggfunc='sum').sort_values(by='amount',ascending=False)

,,amount
col_name,col_location,
maize grain production,CA-QC,0.910290
barley grain production,RoW,0.102624
wheat grain production,RoW,0.072186
"heat and power co-generation, lignite",RoW,0.049769
"operation, housing system, cattle, tied",CA-QC,0.033454
...,...,...
"ammonia production, partial oxidation, liquid",CN,-0.005722
palm fruit bunch production,RoW,-0.007916
"ammonia production, steam reforming, liquid",RoW,-0.009259


Edges LCA

In [10]:
iw_water_edges = ('ImpactWorld+ 2.1', 'Water scarcity', 'midpoint')

In [11]:
iw_water_edges in edges.get_available_methods()

True

In [12]:
milk_qc_edges_lca = edges.EdgeLCIA(demand={milk_qc:1},method=iw_water_edges)
milk_qc_edges_lca.lci()

In [13]:
milk_qc_edges_lca.map_exchanges() # direct mapping
milk_qc_edges_lca.map_aggregate_locations() # e.g. if we have had RER
milk_qc_edges_lca.map_dynamic_locations() # e.g. RoW
milk_qc_edges_lca.map_contained_locations() # e.g. we may have values for CA but not CA-QC
milk_qc_edges_lca.map_remaining_locations_to_global() # default to GLO the unmatched

In [14]:
milk_qc_edges_lca.evaluate_cfs()
milk_qc_edges_lca.lcia()
milk_qc_edges_lca.score

np.float64(0.22765104914561896)

In [15]:
cf_table = milk_qc_edges_lca.generate_cf_table()

In [16]:
cf_table.sort_values(by="impact",ascending=False).head()

,supplier matrix,direction,supplier name,supplier categories,consumer name,consumer reference product,consumer location,consumer cpc,consumer ecospold01categories,consumer isic rev.4 ecoinvent,amount,CF,impact
7471,biosphere,biosphere-technosphere,"Water, turbine use, unspecified natural origin","(natural resource, in water)","electricity production, hydro, run-of-river","electricity, high voltage",RoW,17100: Electrical energy,hydro power/power plants,"3510:Electric power generation, transmission a...",0.145658,38.20,5.564126
1134,biosphere,biosphere-technosphere,"Water, cooling, unspecified natural origin","(natural resource, in water)","heat and power co-generation, lignite","heat, district or industrial, other than natur...",RoW,17300: Steam and hot water,hard coal/power plants,"3510:Electric power generation, transmission a...",0.043332,38.20,1.655271
5258,biosphere,biosphere-technosphere,Water,"(water,)","electricity production, hydro, run-of-river","electricity, high voltage",RU,17100: Electrical energy,hydro power/power plants,"3510:Electric power generation, transmission a...",-0.213642,-7.40,1.580951
7662,biosphere,biosphere-technosphere,"Water, turbine use, unspecified natural origin","(natural resource, in water)","electricity production, hydro, run-of-river","electricity, high voltage",CA-QC,17100: Electrical energy,hydro power/power plants,"3510:Electric power generation, transmission a...",0.546281,1.14,0.622761
5186,biosphere,biosphere-technosphere,Water,"(water,)","electricity production, hydro, run-of-river","electricity, high voltage",IR,17100: Electrical energy,hydro power/power plants,"3510:Electric power generation, transmission a...",-0.007406,-70.00,0.518405


In [17]:
cf_table.pivot_table(index=["consumer location","consumer name"],
                     aggfunc={"CF":"mean",
                              "impact":'sum','amount':"sum"}).sort_values(by="impact",ascending=False)

CF    amount  \
consumer location consumer name                                                
CN                irrigation, surface                    22.300000  0.002307   
IN                irrigation, surface                    36.200000  0.001416   
RoW               heat and power co-generation, lignite   0.000000  0.085487   
                  irrigation, surface                    38.200000  0.001028   
ES                irrigation, surface                    73.500000  0.000341   
...                                                            ...       ...   
RoW               urea production                        12.733333 -0.001105   
ES                barley grain production               -73.500000  0.000170   
US-NE             soybean production                    -33.800000  0.000380   
RoW               wheat grain production                -38.200000  0.000883   
                  barley grain production               -38.200000  0.001667   

                                                           impact  
consumer location consumer name                                    
CN                irrigation, surface                    0.051445  
IN                irrigation, surface                    0.051267  
RoW               heat and power co-generation, lignite  0.044945  
                  irrigation, surface                    0.039256  
ES                irrigation, surface                    0.025077  
...                                                           ...  
RoW               urea production                       -0.010132  
ES                barley grain production               -0.012482  
US-NE             soybean production                    -0.012835  
RoW               wheat grain production                -0.033727  
                  barley grain production               -0.063694  

[3472 rows x 3 columns]

Material Criticality

In [19]:
electric_car = ecoinvent.get(name='transport, passenger, car, electric',
                                location="GLO")
comb_engine = ecoinvent.get(name='transport, passenger, car with internal combustion engine, fleet average',
                               location="RoW")

In [20]:
criticality_lca = edges.EdgeLCIA(demand={electric_car:1,comb_engine:-1},
                                 method=('GeoPolRisk', 'paired', '2024'),)

Method contains 301 duplicate CF matching signature group(s) covering 604 CF entries. These CFs share the same effective CLIPS matching criteria, so only the first matched CF for a given edge will be applied. Examples: indices=[511, 1169] supplier={"categories": null, "classifications": [], "excludes": ["alloy", "liquid", "market"], "location": "GLO", "matrix": "technosphere", "name": "aluminium producti… consumer={"categories": null, "classifications": [], "excludes": [], "location": "BH", "matrix": "technosphere", "name": null, "operator": "equals", "reference product"… values=[0.02225763421839105, 0.0003578840159751286] | indices=[512, 1170] supplier={"categories": null, "classifications": [], "excludes": ["alloy", "liquid", "market"], "location": "GLO", "matrix": "technosphere", "name": "aluminium producti… consumer={"categories": null, "classifications": [], "excludes": [], "location": "BD", "matrix": "technosphere", "name": null, "operator": "equals", "reference product"… values=

In [ ]:
criticality_lca.apply_strategies()
criticality_lca.evaluate_cfs()
criticality_lca.lcia()

Finding direct matches        [1/5]
Resolving aggregate locations [2/5]
Resolving dynamic locations   [3/5]
Resolving contained locations [4/5]
Resolving global locations    [5/5]


In [ ]:
criticality_lca_cfs = criticality_lca.generate_cf_table()

In [ ]:
criticality_lca.score # slightly more for electric (prior to the war with Iran!)

In [ ]:
criticality_lca_cfs.pivot_table(index='supplier reference product',
                                values='impact',
                                aggfunc='sum').sort_values(by='impact',ascending=False)

In [ ]:
criticality_lca_cfs.loc[criticality_lca_cfs['supplier reference product']=='hard coal',
    ['supplier name',"supplier location",'consumer name',"consumer location",'impact']].sort_values(by='impact',
                                                                                                    ascending=False).head(10)

In [ ]:
criticality_lca_cfs.loc[criticality_lca_cfs["supplier reference product"]=="lithium manganese oxide",
                        ["supplier name",'supplier location','consumer name','consumer location','amount','CF','impact']]

In [ ]:
criticality_lca_cfs.loc[criticality_lca_cfs["supplier reference product"]=="cobalt",
                        ["supplier name",'supplier location','consumer name','consumer location','amount','CF','impact']]

In [ ]:
criticality_lca_cfs.pivot_table(index=['supplier reference product','supplier location'],
                                values='impact',
                                aggfunc='sum').sort_values(by='impact',ascending=False)

In [ ]:
criticality_lca.redo_lcia(demand={electric_car.id:1})
criticality_lca.score

In [ ]:
criticality_lca.redo_lcia(demand={comb_engine.id:1})
criticality_lca.score